In [ ]:
import sys
import os

# check that we use the virtual environment
print(sys.version)
print(sys.prefix)
print(sys.executable)

# check if we are in the correct directory
print(os.getcwd())

# switching to the utf-8
!chcp 65001

3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]
c:\Users\PC\Desktop\gpt-dev\venv
c:\Users\PC\Desktop\gpt-dev\venv\Scripts\python.exe
c:\Users\PC\Desktop\gpt-dev
Active code page: 65001


In [3]:
# download the training dataset that we going to use for the model training
# for windows i'll use the curl.exe instaed of wget

# the --ssl-revoke-best-effort option is used to avoid the SSL certificate verification
# -L option is used to follow redirects and -o is used to specify the output file name and folder
!curl.exe --ssl-revoke-best-effort -L -o data/input.txt https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

  % Total    % Received % Xferd  Average Speed  Time    Time    Time   Current
                                 Dload  Upload  Total   Spent   Left   Speed

  0      0   0      0   0      0      0      0                              0
100  1.06M 100  1.06M   0      0  1.65M      0                              0
100  1.06M 100  1.06M   0      0  1.65M      0                              0
100  1.06M 100  1.06M   0      0  1.65M      0                              0


In [5]:
with open('data/input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [6]:
print('length of dataset in characters: ', len(text))

length of dataset in characters:  1115394


In [ ]:
# check the first 667 characters of the dataset
print(text[:667])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they r


In [ ]:
# This text is a sequence of characters. we will use a set() to get unique characters that can appear in a dataset, make a list and sort them.

chars = sorted(list(set(text)))
vocab_size = len(chars)
print('All the unique characters in the dataset: ' + ''.join(chars))
print('Vocabulary size: ', vocab_size)

All the unique characters in the dataset: 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
Vocabulary size:  65


In [ ]:
# Now we need to tokenize the text in our dataset. as Andrej said we could use tiktoken from OpenAI or SentencePiece from Google.
# There is a tradeoff between the vocabulary size and the sequence length of integers.
# They logically use the same approach, but for this notebook we will use simple encoder-decoder approach from Andrej.

# Create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }

# So we take a list of characters and make a lookup table from character to integer, thats one of a many ways to encode
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

print(encode("hii there")) 
print(decode(encode("hii there")))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [20]:
# Now we going to encode the entire text dataset and using pytorch store it into a tensor
# first of all lets download the pytorch library
!pip install torch

  Using cached setuptools-84.0.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ---------------------------------------- 0.0/124.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/124.1 MB ? eta -:--:--
   ---------------------------------------- 0.3/124.1 MB ? eta -:--:--
   ---------------------------------------- 0.3/124.1 MB ? eta -:--:--
   ---------------------------------------- 0.5/124.1 MB 932.9 kB/s eta 0:02:13
   ---------------------------------------- 0.8/124.1 MB 932.9 kB/s eta 0:02:13
   ---------------------------------------- 1.3/124.1 MB 1.2 MB/s eta 0:01:45
    --------------------------------------- 1.6/124.1 MB 1.2 MB/s eta 0:01:41
    --------------------------------------- 1.8/124.1 MB 1.3 MB/s eta 0:01:34
    --------------------------------------- 2.4/124.1 MB 1.4 MB/s 


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import torch # we use the PyTorch

data = torch.tensor(encode(text), dtype=torch.long)

print(data.shape, data.dtype)
print(data[:667]) # the 667 characters we looked at for GPT will look like this. It is an encoded text in a tensor

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

In [ ]:
# now we gonna split our data into train and validation splits
n = int(0.9*len(data)) # here we cut the length of whole data to 90% of its original length

train_data = data[:n] # now we use from 0 to 90% of data for train
val_data = data[n:] # and here we use from 90 till the end for val

In [33]:
# So for training of the transformer we won't use the whole data at once but use the chunks of blocks for that

block_size = 8
train_data[:block_size+1]


tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [ ]:
x = train_data[:block_size]
y = train_data[1:block_size+1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f'When input is {context} the next target: {target}')

# So basically what the transformers are doing is to predict the next token in some sequence

When input is tensor([18]) the next target: 47
When input is tensor([18, 47]) the next target: 56
When input is tensor([18, 47, 56]) the next target: 57
When input is tensor([18, 47, 56, 57]) the next target: 58
When input is tensor([18, 47, 56, 57, 58]) the next target: 1
When input is tensor([18, 47, 56, 57, 58,  1]) the next target: 15
When input is tensor([18, 47, 56, 57, 58,  1, 15]) the next target: 47
When input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the next target: 58


The point of saving encoded text into tensors to feed to GPU. It is because GPU's are good at parallel processing of that

In [ ]:
torch.manual_seed(1337) #random seed to start with

batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)     
print(xb)
print('targets:')
print(yb.shape)     
print(yb)

print('-------------------------')

for b in range(batch_size): # batch dimension
    for t in range(block_size): #time dimension
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f'when input is {context.tolist()} the target: {target}')

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
-------------------------
when input is [24] the target: 43
when input is [24, 43] the target: 58
when input is [24, 43, 58] the target: 5
when input is [24, 43, 58, 5] the target: 57
when input is [24, 43, 58, 5, 57] the target: 1
when input is [24, 43, 58, 5, 57, 1] the target: 46
when input is [24, 43, 58, 5, 57, 1, 46] the target: 43
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target: 39
when input is [44] the target: 53
when input is [44, 53] the target: 56
when input is [44, 53, 56] the target: 1
when input is [44, 53, 56, 1] the target: 58
when input is [44, 53, 56, 1, 58] the target: 46
